# Notebook 06: ML Models - Classification (AQI Bucket)

## Project: Indian Air Quality Index (AQI) Comprehensive Analysis
## BTech Final Year Project - Data Science & Machine Learning (8th Semester)

### Objective:
Train classification models to predict AQI categories (Good, Satisfactory, Moderate, Poor, Very Poor, Severe). Evaluate using accuracy, precision, recall, F1-score, and confusion matrix.

### Prerequisites:
- Complete Notebook 04 (generates X_train, X_test, y_train_clf, y_test_clf)
- Libraries: pandas, numpy, scikit-learn, xgboost, lightgbm

### Run Time: 20-30 minutes

## Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries imported!')

### Explanation:

- **from sklearn.linear_model import LogisticRegression**: Linear classifier for binary/multi-class classification.

- **from sklearn.ensemble import RandomForestClassifier**: Ensemble classifier using multiple decision trees with voting.

- **from sklearn.metrics import accuracy_score**: Percentage of correct predictions.

- **from sklearn.metrics import precision_score**: True positives / (True positives + False positives).

- **from sklearn.metrics import recall_score**: True positives / (True positives + False negatives).

- **from sklearn.metrics import f1_score**: Harmonic mean of precision and recall.

In [ ]:
X_train = pd.read_csv(os.path.join('..', 'datasets', 'X_train.csv'))
X_test = pd.read_csv(os.path.join('..', 'datasets', 'X_test.csv'))
y_train = pd.read_csv(os.path.join('..', 'datasets', 'y_train_clf.csv')).values.ravel()
y_test = pd.read_csv(os.path.join('..', 'datasets', 'y_test_clf.csv')).values.ravel()
print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Classes: {np.unique(y_train)}')

## Step 2: Define Classification Evaluation Function

In [ ]:
def evaluate_classifier(model, X_train, y_train, X_test, y_test, model_name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    print(f'{model_name}:')
    print(f'  Accuracy: {accuracy:.4f}')
    print(f'  Precision: {precision:.4f}')
    print(f'  Recall: {recall:.4f}')
    print(f'  F1-Score: {f1:.4f}')
    print(f'  CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})')
    return {'model': model_name, 'accuracy': accuracy, 'precision': precision, 'recall': recall, 'f1': f1, 'y_pred': y_pred}
print('Evaluation function defined!')

### Explanation:

- **average='weighted'**: Calculates metrics weighted by class support (handles imbalanced classes).

- **cross_val_score(scoring='accuracy')**: Uses accuracy for cross-validation scoring.

## Step 3: Train Logistic Regression

In [ ]:
results = []
lr_results = evaluate_classifier(LogisticRegression(max_iter=1000), X_train, y_train, X_test, y_test, 'Logistic Regression')
results.append(lr_results)

## Step 4: Train Random Forest

In [ ]:
rf_results = evaluate_classifier(RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1), 
                                 X_train, y_train, X_test, y_test, 'Random Forest')
results.append(rf_results)

## Step 5: Train Gradient Boosting

In [ ]:
gb_results = evaluate_classifier(GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42), 
                                 X_train, y_train, X_test, y_test, 'Gradient Boosting')
results.append(gb_results)

## Step 6: Train XGBoost

In [ ]:
xgb_results = evaluate_classifier(XGBClassifier(n_estimators=100, learning_rate=0.1, random_state=42), 
                                  X_train, y_train, X_test, y_test, 'XGBoost')
results.append(xgb_results)

## Step 7: Train LightGBM

In [ ]:
lgbm_results = evaluate_classifier(LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=42), 
                                   X_train, y_train, X_test, y_test, 'LightGBM')
results.append(lgbm_results)

## Step 8: Compare All Models

In [ ]:
results_df = pd.DataFrame(results)
print('\n=== MODEL COMPARISON ===')
print(results_df.to_string(index=False))
best_model = results_df.loc[results_df['accuracy'].idxmax()]
print(f'\nBest Model: {best_model["model"]} (Accuracy = {best_model["accuracy"]:.4f})')

## Step 9: Confusion Matrix (Best Model)

In [ ]:
best_result = results[results_df['accuracy'].idxmax()]
cm = confusion_matrix(y_test, best_result['y_pred'])
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
plt.xlabel('Predicted', fontsize=12, fontweight='bold')
plt.ylabel('Actual', fontsize=12, fontweight='bold')
plt.title(f'Confusion Matrix - {best_result["model"]}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Explanation:

- **confusion_matrix()**: Shows true positives, false positives, true negatives, false negatives for each class.

- **Diagonal elements**: Correct predictions (higher is better).

- **Off-diagonal elements**: Misclassifications (model confusion).

## Step 10: Classification Report

In [ ]:
print(f'\n=== CLASSIFICATION REPORT - {best_result["model"]} ===')
print(classification_report(y_test, best_result['y_pred']))

### Explanation:

- **classification_report()**: Shows precision, recall, f1-score for each class.

- **Support**: Number of actual occurrences of each class in test set.

## Step 11: Save Best Model

In [ ]:
import joblib
best_model_obj = None
for r in results:
    if r['model'] == best_result['model']:
        best_model_obj = r['model']
        break
joblib.dump(best_model_obj, os.path.join('..', 'models', 'best_classification_model.pkl'))
results_df.to_csv(os.path.join('..', 'outputs', 'classification_results.csv'), index=False)
print(f'Best model ({best_result["model"]}) saved to models/ folder!')
print('READY FOR NOTEBOOK 07 (Deep Learning - LSTM)')

## Summary

Trained 5 classification models:
1. Logistic Regression
2. Random Forest
3. Gradient Boosting
4. XGBoost
5. LightGBM

**Metrics:**
- Accuracy: Overall correctness
- Precision: Positive predictive value
- Recall: Sensitivity
- F1-Score: Harmonic mean
- Confusion Matrix: Class-wise performance

**Next**: Notebook 07 - Deep Learning LSTM for Time Series Forecasting